In [5]:
import json
import urllib.request

class SimpleKnowledgeGraph:
    def __init__(self):
        # Adjacency list: { source_node: { target_node: relationship } }
        self.graph = {}

    def add_triple(self, subject, predicate, obj):
        """Adds a directed, labeled edge between two entities."""
        subject = subject.strip().title()
        obj = obj.strip().title()
        predicate = predicate.strip().lower()

        if subject not in self.graph:
            self.graph[subject] = {}
        
        self.graph[subject][obj] = predicate
        
        if obj not in self.graph:
            self.graph[obj] = {}

    def display(self):
        """Prints the entire graph structure."""
        print("\n--- Current Knowledge Graph ---")
        for source, targets in self.graph.items():
            if not targets:
                print(f"({source}) [Leaf Node]")
            for target, rel in targets.items():
                print(f"({source}) --[{rel}]--> ({target})")
    
    def multi_hop_reasoning(self, start_node, target_node):
        """
        System Builder BFS Reasoner:
        Finds a valid chain of reasoning between two nodes across ANY number of hops.
        Tracks a 'visited' set to completely prevent infinite circular loops.
        """
        start_node = start_node.strip().title()
        target_node = target_node.strip().title()
        
        if start_node not in self.graph:
            return f"❌ Entity '{start_node}' does not exist in the graph."

        # Queue stores tuples of: (current_node, path_taken_so_far)
        # Example path: [('Alice', 'works_at'), ('OpenAI', 'is located in')]
        queue = [(start_node, [])]
        visited = set()

        print(f"\n🧠 Executing Multi-Hop Reasoner: Can we link '{start_node}' to '{target_node}'?")

        while queue:
            current_node, path = queue.pop(0)

            if current_node == target_node:
                print(f"✅ Success! Inferred connection found across {len(path)} hops.")
                # Format the logical chain into a scannable sentence
                chain_str = f"({start_node})"
                for next_node, rel in path:
                    chain_str += f" --[{rel}]--> ({next_node})"
                return chain_str

            if current_node not in visited:
                visited.add(current_node)
                
                # Check neighbors
                for neighbor, relationship in self.graph[current_node].items():
                    if neighbor not in visited:
                        # Append the neighbor and the relation used to get there
                        new_path = path + [(neighbor, relationship)]
                        queue.append((neighbor, new_path))

        return f"❌ No logical chain of reasoning could connect '{start_node}' to '{target_node}'."


In [3]:
def extract_triples_with_ollama(text, model_name="llama3.2"):
    url = "http://localhost:11434/api/generate"
    prompt = f"""
    You are a strict Entity-Relationship extraction system. Extract key facts from the text as JSON triples.
    Format your response STRICTLY as a JSON array of objects with keys: "subject", "predicate", "object".
    Do not write any introductory or concluding text. Respond ONLY with raw valid JSON.

    Text: "{text}"
    """
    data = {"model": model_name, "prompt": prompt, "stream": False, "format": "json"}
    req = urllib.request.Request(url, data=json.dumps(data).encode("utf-8"), headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req) as response:
            res_body = json.loads(response.read().decode("utf-8"))
            return json.loads(res_body.get("response", "[]"))
    except Exception as e:
        print(f"Error communicating with local Ollama: {e}")
        return []


In [6]:
kg = SimpleKnowledgeGraph()

    # 1. Simulate pulling dynamic records from llama3.2 text parsing
facts = [
    {"subject": "Sam Altman", "predicate": "ceo_of", "object": "OpenAI"},
    {"subject": "OpenAI", "predicate": "is located in", "object": "San Francisco"},
    {"subject": "San Francisco", "predicate": "part_of", "object": "California"},
    # Let's inject a intentional circular path loop to test safety
    {"subject": "California", "predicate": "contains", "object": "San Francisco"} 
]

print("📥 Ingesting facts into native memory...")
for fact in facts:
    kg.add_triple(fact['subject'], fact['predicate'], fact['object'])

kg.display()

# 2. Run Multi-Hop Reasoner (3 Hops away!)
reasoning_result = kg.multi_hop_reasoning("Sam Altman", "California")
print(f"Logical Proof Chain: {reasoning_result}")

📥 Ingesting facts into native memory...

--- Current Knowledge Graph ---
(Sam Altman) --[ceo_of]--> (Openai)
(Openai) --[is located in]--> (San Francisco)
(San Francisco) --[part_of]--> (California)
(California) --[contains]--> (San Francisco)

🧠 Executing Multi-Hop Reasoner: Can we link 'Sam Altman' to 'California'?
✅ Success! Inferred connection found across 3 hops.
Logical Proof Chain: (Sam Altman) --[ceo_of]--> (Openai) --[is located in]--> (San Francisco) --[part_of]--> (California)
